# Walkthrough: loyalty engagement segmentation

This notebook runs the pipeline step by step on a **small synthetic sample (5,000 users)** so it stays light.
The numbers differ slightly from the README, which uses 200,000 users. All data is synthetic and comes from the
seeded generator in this repository.

Every step is also available from the command line: `python -m loyaltyseg <step>`.

In [1]:
import logging
from pathlib import Path

import pandas as pd

from loyaltyseg import pipeline
from loyaltyseg.config import load_settings, project_root
from loyaltyseg.io import read_json, read_parquet

logging.basicConfig(level=logging.WARNING)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 12)

work = project_root() / "outputs" / "notebook"   # git-ignored
s = load_settings(data_dir=work / "data", output_dir=work / "outputs", seed=42)
print("as of", s.as_of, "| seed", s.seed)

as of 2026-08-31 | seed 42


## 1. Generate synthetic users, catalog and events

In [2]:
meta = pipeline.step_generate(s, n_users=5_000)
meta

{'seed': 42,
 'users': 5000,
 'as_of': '2026-08-31',
 'catalog_items': 150,
 'events': 272962,
 'event_parts': 1}

In [3]:
users = read_parquet(s.raw_dir / "users.parquet")
catalog = read_parquet(s.raw_dir / "catalog.parquet")
users.head(3)

,user_id,msisdn,signup_date,os,push_enabled,in_country,preferred_channel
0,1,+999700000001,2024-06-01,android,True,True,push
1,2,+999700000002,2022-12-01,ios,True,True,email
2,3,+999700000003,2026-06-24,android,True,True,push


## 2. Content clustering of the catalog
K-Means over item features; k is chosen by silhouette and each cluster is named by a rule.

In [4]:
summary = pipeline.step_content(s)
print(read_json(s.output_dir / "content" / "k.json"))
summary

{'best_k': 11, 'elbow_k': 9}


,content_cluster,content_cluster_name,items,dominant_type,dominant_category,median_reward_value,mean_discount_depth,mean_effort
0,0,Daily casual games,12,game,games,2.61,0.000,1.00
1,1,Mixed challenge games,12,game,games,5.53,0.000,2.50
2,2,High-value tech offers,17,offer,tech,64.12,0.317,1.47
3,3,Mid-value shopping offers,17,offer,shopping,33.03,0.309,1.53
4,4,Everyday food offers,17,offer,food,8.15,0.319,1.47
5,5,High-value travel offers,17,offer,travel,73.02,0.331,1.65
6,6,Everyday entertainment offers,17,offer,entertainment,10.75,0.409,1.35
7,7,Mid-value fitness offers,17,offer,fitness,12.74,0.315,1.65
8,8,Tech prize raffles,10,raffle,tech,491.45,0.000,1.00
9,9,Travel prize raffles,7,raffle,travel,580.97,0.000,1.00


## 3. User feature table
Depth-weighted interest shares per content cluster plus recency, frequency, value, discount preference and push response.

In [5]:
features = pipeline.step_features(s)
cols = ["user_id", "loyalty_actions", "loyalty_days_per_30d", "days_since_last_loyalty",
        "discount_pref", "push_open_rate", "share_c0", "share_c2"]
features[cols].head()

,user_id,loyalty_actions,loyalty_days_per_30d,days_since_last_loyalty,discount_pref,push_open_rate,share_c0,share_c2
0,1,14,0.739726,82.579097,0.391967,0.333333,0.000000,0.606061
1,2,0,0.000000,NaN,0.000000,0.111111,0.000000,0.000000
2,3,7,1.739130,49.847211,0.175500,0.500000,0.000000,0.000000
3,4,58,2.547945,137.017882,0.314107,0.000000,0.166667,0.107843
4,5,3,5.454545,3.027454,0.456000,0.000000,0.000000,0.000000


## 4. Archetypes
Browsers (no loyalty actions) are labelled by rule. Engaged users are clustered with MiniBatchKMeans, and k is chosen from silhouette, Davies-Bouldin and stability across 5 seeds.

In [6]:
res = pipeline.step_archetypes(s)
res["k_table"].round(3)

,k,silhouette,davies_bouldin,stability_ari,inertia,stable
0,3,0.179,1.876,0.938,42711.624,True
1,4,0.182,1.754,0.789,39137.463,False
2,5,0.191,1.855,0.848,35582.451,False
3,6,0.220,1.684,0.939,32264.714,True
4,7,0.219,1.706,0.817,30662.770,False
5,8,0.222,1.710,0.745,29235.628,False
6,9,0.227,1.603,0.717,27828.122,False


In [7]:
res["profile"][["archetype", "users", "share", "loyalty_days_per_30d", "recency_days_filled",
                "type_share_game", "cat_share_tech", "cat_share_shopping", "discount_pref"]].round(2)

,archetype,users,share,loyalty_days_per_30d,recency_days_filled,type_share_game,cat_share_tech,cat_share_shopping,discount_pref
cluster,,,,,,,,,
0,Gamers,497,0.10,3.12,6.27,0.74,0.09,0.02,0.29
1,Shoppers,396,0.08,1.66,16.46,0.06,0.06,0.70,0.31
2,Dormant,897,0.18,0.52,197.00,0.20,0.17,0.11,0.34
3,Deal Hunters,438,0.09,1.91,15.64,0.01,0.06,0.10,0.46
4,Occasional Players,728,0.15,0.44,52.07,0.20,0.32,0.03,0.17
5,Tech Chasers,358,0.07,1.71,14.54,0.07,0.75,0.03,0.32
6,Browsers,1686,0.34,0.00,400.00,0.00,0.00,0.00,0.00


Only synthetic data has a ground truth, so this check is not possible in production. There, stability and business interpretability have to do the job.

In [8]:
print("ARI vs planted archetypes:", round(read_json(s.output_dir / "archetypes" / "evaluation.json")["ari_vs_planted"], 3))
pd.read_csv(s.output_dir / "archetypes" / "confusion_vs_planted.csv").set_index("archetype")

ARI vs planted archetypes: 0.878


,Browsers,Deal Hunters,Dormant,Gamers,Occasional Players,Shoppers,Tech Chasers
archetype,,,,,,,
Browsers,1686,0,0,0,0,0,0
Deal Hunters,0,401,16,1,6,8,6
Dormant,0,28,694,7,103,36,29
Gamers,0,2,0,461,22,5,7
Occasional Players,0,3,35,19,638,10,23
Shoppers,0,9,1,1,0,383,2
Tech Chasers,0,3,10,2,5,2,336


## 5. Five-layer segmentation and opportunity sizing

In [9]:
master = pipeline.step_segments(s)
master.head()

,user_id,archetype,lifecycle,interest,play_style,deal_style,channel,reachable
0,1,Tech Chasers,at-risk,High-value tech offers,one-and-done,any-deal,reachable-responsive,True
1,2,Browsers,browser,none,non-player,non-redeemer,reachable-responsive,True
2,3,Gamers,at-risk,Mid-value shopping offers,one-and-done,non-redeemer,reachable-unresponsive,True
3,4,Dormant,dormant,mixed,collector,any-deal,reachable-unresponsive,True
4,5,Gamers,new,Travel prize raffles,one-and-done,non-redeemer,reachable-unresponsive,True


In [10]:
sizes = pd.read_csv(s.output_dir / "segments" / "layer_sizes.csv")
sizes[sizes["layer"] == "lifecycle"].round(3)

,layer,label,users,reachable_users,share_of_base,reachable_rate,share_of_reachable
7,lifecycle,browser,1686,604,0.337,0.358,0.334
8,lifecycle,active,1405,724,0.281,0.515,0.400
9,lifecycle,dormant,1122,176,0.224,0.157,0.097
10,lifecycle,at-risk,687,255,0.137,0.371,0.141
11,lifecycle,new,100,51,0.020,0.510,0.028


In [11]:
read_json(s.output_dir / "segments" / "browsers.json")

{'browsers': 1686,
 'browser_share_of_base': 0.3372,
 'reachable_browsers': 604,
 'browser_reachable_rate': 0.3582443653618031,
 'browser_share_of_reachable': 0.33370165745856356,
 'base_users': 5000,
 'reachable_users': 1810}

## 6. Campaign audiences
The filters run in this order: reachability, dedupe, exclusion of users contacted in the last 7 days, then the size cap. Each step is counted in the manifest.

In [12]:
manifest = pipeline.step_audiences(s)
pd.DataFrame([{"audience": a["audience"], **a["counts"]} for a in manifest["audiences"]])

,audience,matched,after_reachability,after_dedupe,after_exclusion,after_cap
0,reachable_browsers,1686,604,604,571,571
1,at_risk_deal_hunters,110,46,46,43,43
2,tech_chasers,326,170,170,163,163
3,shoppers,365,168,168,156,156
4,active_gamers,409,219,219,212,212
5,high_value_redeemers,387,192,24,18,18


## 7. Experiment readout: broad vs targeted (simulated)
The outcome is driven by the generator's latent parameters. It shows how the analysis framework works and is **not** evidence on its own.

In [13]:
ro = pipeline.step_experiment(s)
pd.DataFrame({k: ro[k] for k in ("acquisition", "retention")}).loc[
    ["n_a", "p_a", "n_b", "p_b", "diff", "ci_low", "ci_high", "p_value"]]

,acquisition,retention
n_a,526,74
p_a,0.032319,0.445946
n_b,509,134
p_b,0.084479,0.559701
diff,0.05216,0.113756
ci_low,0.023662,-0.02728
ci_high,0.080658,0.254791
p_value,0.000331,0.115944


In [14]:
ro["planning"]

{'baseline_acquisition_rate': 0.03231939163498099,
 'relative_lift_to_detect': 0.1,
 'required_n_per_arm': 49268,
 'current_n_per_arm': 509,
 'mde_absolute_at_current_n': 0.03885655036102358,
 'alpha': 0.05,
 'power': 0.8}

## 8. PDF report

In [15]:
pdf = pipeline.step_report(s)
print(pdf.relative_to(project_root()))

outputs\notebook\outputs\report\loyalty_segmentation_report.pdf
